# Keypoint formats -> YOLO Pose workflow

Notebook pokazuje, jak pracowac na danych np. w COCO i korzystac z istniejacych metod projektu (YPSetValidation, MergedRankingEvaluator) bez przepisywania calego pipeline'u.

## Po co KeypointDatasetAdapter

Adapter jest warstwa zgodnosci:
- bierze dane z innego formatu (np. COCO),
- konwertuje je do struktury YOLO Pose,
- zwraca obiekt zgodny z `YOLOPoseDataset`,
- dzieki temu mozna od razu uzyc wszystkich Twoich metod bez zmian w logice notebookow.

Najwieksza zaleta: ten sam kod analizy dziala dla wielu formatow danych keypointowych.

In [ ]:
from pathlib import Path

from utils.datasets import KeypointDatasetAdapter
from utils.validation import YPSetValidation, MergedRankingEvaluator

## Konfiguracja

In [ ]:
# MODE: "coco" albo "registered"
MODE = "coco"

# COCO
COCO_ANNOTATIONS_DIR = r"D:\\data\\coco\\annotations"
COCO_IMAGES_DIR = r"D:\\data\\coco\\images"
COCO_USE_ULTRALYTICS = True

# Registered format (FORMAT_REGISTRY)
REGISTERED_FORMAT_NAME = "coco_keypoints"  # np. "yolo_pose", "coco_keypoints"
REGISTERED_INPUT_ROOT = r"D:\\data\\coco_root"
REGISTERED_COPY_IMAGES = True

# Wspolne
OUTPUT_ROOT = None  # None => katalog tymczasowy i auto-cleanup
SPLIT_NAME = "train"

# Optional evaluator
RUN_EVALUATOR = False
REFERENCE_REPORT_PATHS = {
    "merged": r"D:\\data\\reference\\results.txt",
}

In [ ]:
def build_dataset_adapter():
    if MODE == "coco":
        return KeypointDatasetAdapter.from_coco(
            annotations_dir=COCO_ANNOTATIONS_DIR,
            images_dir=COCO_IMAGES_DIR,
            output_root=OUTPUT_ROOT,
            split_name=SPLIT_NAME,
            use_ultralytics=COCO_USE_ULTRALYTICS,
        )

    if MODE == "registered":
        return KeypointDatasetAdapter.from_registered_format(
            format_name=REGISTERED_FORMAT_NAME,
            input_root=REGISTERED_INPUT_ROOT,
            output_root=OUTPUT_ROOT,
            split_name=SPLIT_NAME,
            copy_images=REGISTERED_COPY_IMAGES,
        )

    raise ValueError("MODE musi byc jednym z: 'coco', 'registered'")

## Uzycie jak YOLOPoseDataset

In [ ]:
with build_dataset_adapter() as dataset:
    print("=== ADAPTER INFO ===")
    for key, value in dataset.info().items():
        print(f"{key}: {value}")

    print("\n=== PODSTAWOWE STATYSTYKI ===")
    print(f"Liczba obrazow: {len(dataset)}")
    first = dataset[0]
    print(f"Pierwszy obraz: {first['image_path']}")
    print(f"Pierwszy sample person_count: {first.get('person_count', 'n/a')}")

    validator = YPSetValidation(dataset)
    stats = validator.get_stats()
    print("\n=== YPSetValidation.get_stats() ===")
    print(stats)

    lbp_decimals, lbp_counts, lbp_paths = validator.get_all_lbp_decimals(
        patch_size=5,
        verbose=False,
        visibility_threshold=2.0,
    )
    print("\n=== LBP DECIMALS ===")
    print(f"Liczba wszystkich decimali: {len(lbp_decimals)}")
    print(f"Liczba obrazow zliczonych: {len(lbp_counts)}")
    print(f"Pierwsze 3 sciezki: {lbp_paths[:3]}")

    if RUN_EVALUATOR:
        missing = [name for name, path in REFERENCE_REPORT_PATHS.items() if not Path(path).exists()]
        if missing:
            raise FileNotFoundError(
                f"Brakuje plikow referencyjnych dla: {missing}. Ustaw REFERENCE_REPORT_PATHS poprawnie."
            )

        evaluator = MergedRankingEvaluator(
            validator=validator,
            report_paths=REFERENCE_REPORT_PATHS,
            top_k=450,
            segmentation_model_name=None,
        )
        result = evaluator.run(1.0, 1.0, 0.0)
        print("\n=== EVALUATOR ===")
        print(f"Liczba rekordow rankingu: {len(result['combined_ranking']) if result else 0}")